## 5. Merge com variáveis macroeconômicas

Incorporamos dois grupos de variáveis macro:

**Nível** (valor no mês t): SELIC, IPCA, desemprego, saldo de crédito do SFN, índice de confiança

**Tendência** (variação em relação a meses anteriores): capturam se o ambiente macro está melhorando
ou piorando — informação essencial para prever seis meses à frente.

| Feature | Cálculo | Significado |
|---|---|---|
| `selic_delta3` | SELIC(t) − SELIC(t−3) | SELIC subindo ou caindo? |
| `selic_delta6` | SELIC(t) − SELIC(t−6) | Tendência semestral dos juros |
| `ipca_roll6_mean` | média IPCA últimos 6m | Inflação média recente |
| `desemprego_delta3` | Desemp(t) − Desemp(t−3) | Desemprego acelerando? |
| `spread_delta3` | Saldo(t) − Saldo(t−3) | Crédito expandindo ou contraindo? |
| `confianca_delta3` | Conf(t) − Conf(t−3) | Confiança melhorando? |

> **Nota de nomenclatura.** A coluna chamada `spread_pj` **não é um spread**: a série SGS 20539
> devolve um *saldo* de crédito em R$ milhões. O nome foi mantido para preservar a
> reprodutibilidade dos resultados já publicados, mas sua leitura correta é **volume de crédito**,
> não custo do crédito. Ver a nota de proveniência no notebook `00_coleta`.

In [ ]:
import pandas as pd
import numpy as np
import json
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", "{:.4f}".format)

BASE = Path().resolve().parent
PROC = BASE / "data" / "processed"
PARQ = PROC / "scr_pj"

print("Base:", BASE)
print("Parquets disponíveis:", len(list(PARQ.glob("*.parquet"))))


## 1. Carga e agregação

Carregamos apenas as colunas necessárias (2012–2024) e agregamos para o nível `(data_base, cnae_secao, porte, modalidade)` — eliminando dimensões como UF, SR e indexador que aumentariam esparsidade sem ganho preditivo relevante.

In [ ]:
COLUNAS = [
    "data_base", "cnae_secao", "porte", "modalidade",
    "carteira_ativa", "carteira_inadimplida_arrastada", "vencido_acima_de_15_dias",
]
GROUP_KEYS = ["data_base", "cnae_secao", "porte", "modalidade"]

# Exclui 2025 (ano incompleto)
arquivos = sorted(f for f in PARQ.glob("*.parquet") if "2025" not in f.name)
print(f"Carregando {len(arquivos)} arquivos (2012–2024)...\n")

chunks = []
for f in arquivos:
    df_tmp = pd.read_parquet(f, columns=COLUNAS)
    chunks.append(df_tmp)
    print(f"  {f.name}: {len(df_tmp):>10,} linhas")

df_raw = pd.concat(chunks, ignore_index=True)
del chunks
print(f"\nTotal bruto: {len(df_raw):,} linhas")

# Agregação
df_agg = (
    df_raw
    .groupby(GROUP_KEYS, observed=True)
    .agg(
        carteira_ativa=("carteira_ativa", "sum"),
        carteira_inadimplida=("carteira_inadimplida_arrastada", "sum"),
        vencido=("vencido_acima_de_15_dias", "sum"),
    )
    .reset_index()
)
del df_raw

# Taxas
df_agg["taxa_inadimplencia"] = df_agg["carteira_inadimplida"] / df_agg["carteira_ativa"]
df_agg["taxa_vencido"]       = df_agg["vencido"] / df_agg["carteira_ativa"]

print(f"\nShape pós-agregação:       {df_agg.shape}")
print(f"Segmentos únicos:          {df_agg.groupby(['cnae_secao','porte','modalidade']).ngroups}")
print(f"Meses únicos:              {df_agg['data_base'].nunique()}")
print(f"\ntaxa_inadimplencia — describe:")
print(df_agg["taxa_inadimplencia"].describe())

## 2. Filtro de segmentos esparsos

Removemos linhas com carteira ativa < R$10M (ruído estatístico) e segmentos que aparecem em menos de 80% dos meses disponíveis (descontinuidade temporal compromete os lags).

In [ ]:
# Filtro 1: carteira mínima
MIN_CARTEIRA = 10_000_000  # R$10M
before = len(df_agg)
df_agg = df_agg[df_agg["carteira_ativa"] >= MIN_CARTEIRA].copy()
print(f"Filtro carteira < R$10M: {before - len(df_agg):,} linhas removidas → {len(df_agg):,} restantes")

# Filtro 2: continuidade temporal (>= 80% dos meses)
total_months = df_agg["data_base"].nunique()
min_months   = int(total_months * 0.80)

presenca = (
    df_agg.groupby(["cnae_secao", "porte", "modalidade"])["data_base"]
    .nunique()
    .reset_index(name="n_meses")
)
segmentos_ativos = presenca[presenca["n_meses"] >= min_months][["cnae_secao", "porte", "modalidade"]]

print(f"\nTotal de meses: {total_months} | Mínimo exigido: {min_months}")
print(f"Segmentos antes: {presenca.shape[0]} | Após filtro: {len(segmentos_ativos)}")

df_agg = df_agg.merge(segmentos_ativos, on=["cnae_secao", "porte", "modalidade"], how="inner")
print(f"\nShape final:  {df_agg.shape}")

## 3. Features de lag e janela temporal

> **Revisão A1 — defasagem de publicação.** Os dados do SCR referentes ao mês `t` são publicados
> pelo BCB cerca de **2 meses** depois (verificado em 10/11/2026: o ZIP de 2026 continha dados até
> setembro). Na data em que a previsão é feita (origem `t`), o último mês *publicado* é `t−2`.
> Por isso todas as variáveis derivadas do SCR são calculadas sobre a série **disponível**
> (`nivel_disp = taxa(t − LAG_SCR)`), e não sobre a taxa real em `t`, que ainda não é conhecida.

Para cada segmento, criamos (sempre a partir do último valor publicado):
- **Nível disponível**: `nivel_disp` — última taxa publicada na origem
- **Lags**: `inadim_lag{k}` = taxa em `t − LAG_SCR − k`, k ∈ {1, 2, 3, 6, 12}
- **Médias móveis**: janelas de 3, 6, 12 meses sobre a série disponível
- **Desvio padrão móvel**: volatilidade nos últimos 3 e 6 meses; e `vol12_disp` (12 meses,
  incluindo o último valor publicado) — referencial de volatilidade usado na revisão A2a
- **Deltas**: momentum e tendência da série disponível
- **Taxa de vencido**: proporção de crédito vencido, defasada
- **Crescimento da carteira**: variação percentual da carteira ativa publicada
- **Referenciais sazonais** (revisão A4b): `taxa_m6 = taxa(t−6)` e `taxa_m12 = taxa(t−12)`,
  valores reais já publicados na origem, usados apenas como baselines no notebook 04

In [ ]:
SEG_KEYS = ["cnae_secao", "porte", "modalidade"]

# --- Defasagem de publicacao do SCR (revisao A1) ---
# O mes t do SCR e publicado ~2 meses depois. Na origem t, o ultimo mes PUBLICADO e t - LAG_SCR.
# Toda variavel derivada do SCR e calculada sobre a serie disponivel (deslocada em LAG_SCR).
LAG_SCR = 2

df_panel = df_agg.sort_values(SEG_KEYS + ["data_base"]).copy()

def grp_shift(df, col, n):
    return df.groupby(SEG_KEYS, observed=True)[col].shift(n)

def grp_rolling(df, col, window, agg="mean", shift=1):
    return (
        df.groupby(SEG_KEYS, observed=True)[col]
        .transform(lambda x: x.shift(shift).rolling(window, min_periods=max(1, window // 2)).agg(agg))
    )

# Series DISPONIVEIS na origem (ultimo valor publicado)
df_panel["nivel_disp"]    = grp_shift(df_panel, "taxa_inadimplencia", LAG_SCR)
df_panel["vencido_disp"]  = grp_shift(df_panel, "taxa_vencido", LAG_SCR)
df_panel["carteira_disp"] = grp_shift(df_panel, "carteira_ativa", LAG_SCR)

# Lags da taxa de inadimplencia (lag k = taxa em t - LAG_SCR - k)
for lag in [1, 2, 3, 6, 12]:
    df_panel[f"inadim_lag{lag}"] = grp_shift(df_panel, "nivel_disp", lag)

# Medias e desvio padrao movel (sobre a serie disponivel)
for w in [3, 6, 12]:
    df_panel[f"inadim_roll{w}_mean"] = grp_rolling(df_panel, "nivel_disp", w, "mean")
for w in [3, 6]:
    df_panel[f"inadim_roll{w}_std"]  = grp_rolling(df_panel, "nivel_disp", w, "std")

# Volatilidade de 12 meses INCLUINDO o ultimo valor publicado (referencial A2a) — nao e feature
df_panel["vol12_disp"] = grp_rolling(df_panel, "nivel_disp", 12, "std", shift=0)

# Deltas (momentum) da serie disponivel
df_panel["inadim_delta1"] = grp_shift(df_panel, "nivel_disp", 1) - grp_shift(df_panel, "nivel_disp", 2)
df_panel["inadim_delta3"] = grp_shift(df_panel, "nivel_disp", 1) - grp_shift(df_panel, "nivel_disp", 4)

# Taxa de vencido defasada (sobre a serie disponivel)
df_panel["vencido_lag1"] = grp_shift(df_panel, "vencido_disp", 1)

# Crescimento da carteira ativa publicada
df_panel["carteira_growth1"] = (
    df_panel.groupby(SEG_KEYS, observed=True)["carteira_disp"]
    .transform(lambda x: x.pct_change(1).shift(1))
)

# Referenciais sazonais ingenuos (revisao A4b): taxa(t-6) e taxa(t-12), ja publicadas na origem.
# Nao sao features — servem apenas como baselines no notebook 04.
df_panel["taxa_m6"]  = grp_shift(df_panel, "taxa_inadimplencia", 6)
df_panel["taxa_m12"] = grp_shift(df_panel, "taxa_inadimplencia", 12)

print(f"Defasagem de publicacao do SCR: {LAG_SCR} meses -> nivel conhecido na origem = taxa(t-{LAG_SCR})\n")
print("Features de lag criadas:")
lag_cols = [c for c in df_panel.columns
            if c.startswith(("nivel_disp", "inadim_", "vencido_lag", "carteira_growth", "vol12", "taxa_m"))]
for c in lag_cols:
    nulos = df_panel[c].isna().mean()
    print(f"  {c:<30} nulos: {nulos:.2%}")

## 4. Variável alvo

**Definição**: o target é **contínuo** — a taxa de inadimplência real do segmento daqui a 6 meses (t+6).

Optamos por regressão em vez de classificação binária pois:
1. O TCC prevê **métricas de erro** (RMSE, MAE) alinhadas ao documento acadêmico aprovado;
2. A pergunta real é *quanto* a inadimplência vai ser, não apenas *se* vai ser alta;
3. Facilita comparação com o baseline ARIMA (modelo estatístico de série temporal).

O target é calculado com `shift(-6)` por segmento, garantindo que o modelo aprenda a prever
**6 meses à frente** sem vazamento de dados (os últimos 6 meses de cada série ficam sem target e são removidos).

In [ ]:
# Inadimplencia daqui a 6 meses - target continuo para regressao
HORIZONTE = 6
SPLIT_DATE = pd.Timestamp("2022-01-01")

df_panel[f"inadim_t{HORIZONTE}"] = (
    df_panel.groupby(SEG_KEYS, observed=True)["taxa_inadimplencia"].shift(-HORIZONTE)
)

# Target continuo: taxa de inadimplencia em t+6
df_panel["target"] = df_panel[f"inadim_t{HORIZONTE}"]

# Diagnostico
valid = df_panel["target"].notna()
print(f"Horizonte de previsao: t+{HORIZONTE} meses")
print(f"Linhas com target valido: {valid.sum():,} / {len(df_panel):,}")
print("")
print("Estatisticas do target (taxa de inadimplencia em t+6):")
print(df_panel.loc[valid, "target"].describe().round(4))
print("")
print("Distribuicao por quantil:")
q = df_panel.loc[valid, "target"]
for p in [0.1, 0.25, 0.5, 0.75, 0.9, 0.95]:
    print(f"  P{int(p*100):3d}: {q.quantile(p):.4f}")


## 5. Merge com variáveis macroeconômicas

> **Revisão A1 — defasagem de publicação das séries macro.** Cada série entra com a defasagem
> com que de fato fica disponível (verificada na API do SGS em 10/11/2026): Selic 0 meses,
> IPCA 1, confiança 1, desemprego 2, spread e saldos de crédito 2, IBC-Br 2. As variáveis de
> tendência (deltas e médias móveis) são calculadas **depois** do deslocamento.

Variáveis de **nível**: SELIC meta, IPCA, desemprego, `spread_pj`, índice de confiança.
Variáveis de **tendência**: deltas de 3 e 6 meses e média móvel de 6 meses.

> **Revisão A4c.** Quando o `macro.parquet` traz o esquema revisado do notebook `00`, `spread_pj`
> passa a ser o **spread médio PJ de fato** (SGS 20783) e entram duas variáveis adicionais em
> **variação de 12 meses**: `ibcbr_var12` (IBC-Br dessazonalizado, SGS 24364) e `credito_var12`
> (saldo de crédito do SFN, SGS 20539 — a série que antes era chamada, incorretamente, de
> `spread_pj`). Níveis nominais não entram no modelo em nível. Com o esquema antigo, o código
> reproduz exatamente o conjunto original de 28 variáveis.

In [ ]:
macro = pd.read_parquet(PROC / "macro.parquet")
macro["data_mes"] = pd.to_datetime(macro["data"]).dt.to_period("M").dt.to_timestamp("M")
macro = macro.sort_values("data_mes").reset_index(drop=True)

# --- Defasagens de publicacao das series macro, em meses (revisao A1) ---
# Verificadas na API do SGS em 10/11/2026 (ultima observacao disponivel vs mes corrente).
LAGS_MACRO = {
    "selic_meta": 0, "ipca": 1, "desemprego": 2, "spread_pj": 2,
    "confianca_empresarial": 1, "ibcbr_dessaz": 2, "credito_saldo_sfn": 2,
}
for col, k in LAGS_MACRO.items():
    if col in macro.columns and k > 0:
        macro[col] = macro[col].shift(k)

# Esquema A4c (notebook 00 revisado): spread_pj = SGS 20783 (spread PJ de fato) e IBC-Br presente.
A4C = "ibcbr_dessaz" in macro.columns
print(f"Esquema macro: {'A4c (spread PJ real + IBC-Br)' if A4C else 'original'}")
print(f"Defasagens aplicadas: { {k: v for k, v in LAGS_MACRO.items() if k in macro.columns} }")

# Features de nivel
MACRO_BASE = ["selic_meta", "ipca", "desemprego", "spread_pj", "confianca_empresarial"]

# Features de tendencia (calculadas DEPOIS do deslocamento)
macro["selic_delta3"]       = macro["selic_meta"].diff(3)
macro["selic_delta6"]       = macro["selic_meta"].diff(6)
macro["ipca_roll6_mean"]    = macro["ipca"].rolling(6, min_periods=3).mean()
macro["desemprego_delta3"]  = macro["desemprego"].diff(3)
macro["spread_delta3"]      = macro["spread_pj"].diff(3)
macro["confianca_delta3"]   = macro["confianca_empresarial"].diff(3)

MACRO_TREND = [
    "selic_delta3", "selic_delta6",
    "ipca_roll6_mean",
    "desemprego_delta3",
    "spread_delta3",
    "confianca_delta3",
]

# Revisao A4c: niveis nominais entram apenas em VARIACAO de 12 meses
MACRO_EXTRA = []
if A4C:
    macro["ibcbr_var12"]   = macro["ibcbr_dessaz"].pct_change(12) * 100
    macro["credito_var12"] = macro["credito_saldo_sfn"].pct_change(12) * 100
    MACRO_EXTRA = ["ibcbr_var12", "credito_var12"]

MACRO_COLS = MACRO_BASE + MACRO_TREND + MACRO_EXTRA

df_panel["data_mes"] = df_panel["data_base"].dt.to_period("M").dt.to_timestamp("M")
df_panel = df_panel.merge(
    macro[["data_mes"] + MACRO_COLS],
    on="data_mes",
    how="left"
)
df_panel = df_panel.drop(columns=["data_mes"])

print(f"\nShape pos-merge macro: {df_panel.shape}")
print(f"Features macro ({len(MACRO_COLS)}):")
for c in MACRO_COLS:
    nulos = df_panel[c].isna().sum()
    print(f"  {c:<25} nulos: {nulos}")

## 6. Encoding de variáveis categóricas

Label Encoding para `cnae_secao`, `porte` e `modalidade`. Modelos baseados em árvore (RF, XGBoost)
funcionam bem com label encoding. Nos notebooks 04 e 07, os modelos lineares (Regressão Linear e
Logística) são avaliados também em uma variante com **One-Hot Encoding + padronização**
(revisão A4a), para que a comparação com os modelos de árvore não seja prejudicada por uma
codificação ordinal arbitrária.

In [ ]:
# Remove prefixo "PJ - " para limpeza
for col in ["cnae_secao", "porte", "modalidade"]:
    df_panel[col] = df_panel[col].str.replace("PJ - ", "", regex=False).str.strip()

# Label encoding + registro do mapeamento
encoding_maps = {}
for col in ["cnae_secao", "porte", "modalidade"]:
    le = LabelEncoder()
    df_panel[f"{col}_enc"] = le.fit_transform(df_panel[col].astype(str))
    encoding_maps[col] = dict(zip(le.classes_, le.transform(le.classes_).tolist()))
    print(f"\n{col}: {encoding_maps[col]}")

## 7. Dataset final e split temporal

Selecionamos as features, removemos linhas com NaN (primeiros meses sem histórico suficiente e **últimos 6 meses sem target conhecido**) e aplicamos o split temporal:

- **Treino**: 2013-01 → 2021-12 (9 anos)
- **Teste**: 2022-01 → 2024-06 (os últimos 6 meses de 2024 não têm target)

In [ ]:
FEATURE_COLS = [
    # Lags
    "inadim_lag1", "inadim_lag2", "inadim_lag3", "inadim_lag6", "inadim_lag12",
    # Rolling
    "inadim_roll3_mean", "inadim_roll6_mean", "inadim_roll12_mean",
    "inadim_roll3_std",  "inadim_roll6_std",
    # Momentum
    "inadim_delta1", "inadim_delta3",
    # Vencido e carteira
    "vencido_lag1", "carteira_growth1",
    # Macro — nível
    "selic_meta", "ipca", "desemprego",
    "spread_pj", "confianca_empresarial",
    # Macro — tendência (delta e média móvel)
    "selic_delta3", "selic_delta6",
    "ipca_roll6_mean",
    "desemprego_delta3",
    "spread_delta3",
    "confianca_delta3",
    # Categóricas (encoded)
    "cnae_secao_enc", "porte_enc", "modalidade_enc",
]
if A4C:   # revisao A4c: atividade e credito em variacao de 12 meses
    FEATURE_COLS = FEATURE_COLS + ["ibcbr_var12", "credito_var12"]

TARGET_COL = "target"  # taxa de inadimplencia em t+6 (regressao continua)
# Colunas de apoio (nao sao features): nivel conhecido na origem, volatilidade (A2a),
# referenciais sazonais (A4b) e carteira ativa real em t (ponderacao, A3).
META_COLS  = ["data_base", "cnae_secao", "porte", "modalidade", "taxa_inadimplencia",
              "nivel_disp", "vol12_disp", "taxa_m6", "taxa_m12", "carteira_ativa"]

df_ml = (df_panel[META_COLS + FEATURE_COLS + [TARGET_COL]]
         .dropna(subset=FEATURE_COLS + [TARGET_COL] + ["nivel_disp"])
         .copy())

print(f"Dataset ML:")
print(f"  Shape:     {df_ml.shape}")
print(f"  Features:  {len(FEATURE_COLS)}")
print(f"  Período:   {df_ml['data_base'].min().date()} → {df_ml['data_base'].max().date()}")
print(f"  Target (média):  {df_ml[TARGET_COL].mean():.4f}")
for c in ["vol12_disp", "taxa_m6", "taxa_m12", "carteira_ativa"]:
    print(f"  nulos em {c:<16}: {df_ml[c].isna().sum()}")

# Split temporal
df_train = df_ml[df_ml["data_base"] <  SPLIT_DATE]
df_test  = df_ml[df_ml["data_base"] >= SPLIT_DATE]

X_train = df_train[FEATURE_COLS]
y_train = df_train[TARGET_COL]
X_test  = df_test[FEATURE_COLS]
y_test  = df_test[TARGET_COL]

print(f"\nTreino: {len(df_train):,} linhas  ({df_train['data_base'].min().date()} → {df_train['data_base'].max().date()})  target_mean: {y_train.mean():.4f}")
print(f"Teste:  {len(df_test):,}  linhas  ({df_test['data_base'].min().date()} → {df_test['data_base'].max().date()})  target_mean: {y_test.mean():.4f}")

## 8. Análise exploratória das features

Antes de modelar, verificamos:
- **Heatmap de correlação** entre features — detecta multicolinearidade
- **Correlação com o target** — identifica as features mais preditivas
- **Distribuição** das features numéricas — detecta outliers e assimetria

In [ ]:
import seaborn as sns

df_eda = df_ml[FEATURE_COLS + [TARGET_COL]].copy()

# Heatmap de correlacao entre features
corr = df_eda[FEATURE_COLS].corr()

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=False, cmap="RdBu_r",
    center=0, vmin=-1, vmax=1, linewidths=0.3, ax=ax,
    xticklabels=FEATURE_COLS, yticklabels=FEATURE_COLS,
)
ax.set_title("Heatmap de correlacao linear entre features", fontsize=13, pad=15)
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig(PROC / "eda_heatmap_correlacao.png", dpi=150, bbox_inches="tight")
plt.show()
print("Salvo: eda_heatmap_correlacao.png")

# Pares com alta correlacao
print("\n=== Pares com |correlacao| > 0.85 ===")
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
high_corr = (
    upper.stack()
    .reset_index()
    .rename(columns={"level_0": "Feature A", "level_1": "Feature B", 0: "r"})
    .assign(abs_r=lambda x: x["r"].abs())
    .query("abs_r > 0.85")
    .sort_values("abs_r", ascending=False)
)
if len(high_corr):
    print(high_corr[["Feature A", "Feature B", "r", "abs_r"]].to_string(index=False))
else:
    print("Nenhum par acima de 0.85")


In [ ]:
# Correlacao de cada feature com o target
corr_target = (
    df_eda.corr()[TARGET_COL]
    .drop(TARGET_COL)
    .sort_values(key=abs, ascending=False)
    .reset_index()
    .rename(columns={"index": "Feature", TARGET_COL: "r_target"})
)

fig, ax = plt.subplots(figsize=(8, 8))
colors = ["#d73027" if v > 0 else "#4575b4" for v in corr_target["r_target"]]
ax.barh(corr_target["Feature"], corr_target["r_target"], color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Correlacao de cada feature com o target (t+6)", fontsize=12)
ax.set_xlabel("Correlacao de Pearson")
ax.grid(True, axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(PROC / "eda_correlacao_target.png", dpi=150, bbox_inches="tight")
plt.show()

print("Top 10 features mais correlacionadas com o target:")
print(corr_target.head(10).to_string(index=False))


In [ ]:
# Distribuicao das features numericas
numericas = [c for c in FEATURE_COLS if c not in ["cnae_secao_enc", "porte_enc", "modalidade_enc"]]

ncols = 5
nrows = (len(numericas) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*3.5, nrows*2.8))
axes = axes.flatten()

for i, col in enumerate(numericas):
    axes[i].hist(df_eda[col].dropna(), bins=40, color="#377eb8", alpha=0.7, edgecolor="none")
    axes[i].set_title(col, fontsize=8)
    axes[i].tick_params(labelsize=7)
    axes[i].grid(True, alpha=0.2)

for j in range(len(numericas), len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Distribuicao das features numericas", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(PROC / "eda_distribuicoes.png", dpi=150, bbox_inches="tight")
plt.show()

print("Estatisticas descritivas:")
print(df_eda[numericas].describe().round(4).T.to_string())


In [ ]:
# Salva dataset completo
OUT_DATASET  = PROC / "ml_dataset.parquet"
OUT_METADATA = PROC / "ml_metadata.json"

df_ml.to_parquet(OUT_DATASET, index=False)

metadata = {
    "feature_cols":   FEATURE_COLS,
    "target_col":     TARGET_COL,
    "tipo_target":    "regressao_continua",
    "horizonte":      HORIZONTE,
    # Revisao A1: defasagens de publicacao e embargo resultante
    "lag_scr":        LAG_SCR,
    "lags_macro":     {k: v for k, v in LAGS_MACRO.items() if k in macro.columns},
    "embargo_meses":  HORIZONTE + LAG_SCR,
    "nivel_col":      "nivel_disp",
    "macro_esquema":  "a4c" if A4C else "original",
    "split_date":     str(SPLIT_DATE.date()),
    "n_train":        int(len(df_train)),
    "n_test":         int(len(df_test)),
    "n_features":     int(len(FEATURE_COLS)),
    "target_mean_train": float(y_train.mean()),
    "target_mean_test":  float(y_test.mean()),
}

with open(OUT_METADATA, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print(f"Dataset salvo:   {OUT_DATASET}  ({OUT_DATASET.stat().st_size / 1e6:.1f} MB)")
print(f"Metadados:       {OUT_METADATA}")
print(f"  lag_scr={LAG_SCR}  embargo={HORIZONTE + LAG_SCR}  esquema macro={metadata['macro_esquema']}")
print(f"\nPronto para modelagem! → 04_modelos.ipynb")